In [23]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
import os
from dotenv import load_dotenv

In [24]:
load_dotenv()


True

In [25]:
GROK_API_KEY = os.getenv("GROK_API_KEY")

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    temperature=0,
    api_key=GROK_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    # Groq free tier caps OUTPUT at 1000 tokens/minute (OTPM). Keep every
    # single response under that so a call isn't rejected before it runs.
    max_tokens=900,
)

In [26]:
class LLMState(TypedDict):
    question:str
    answer:str

In [27]:
def llm_qa(state:LLMState)->LLMState:
    question=state['question']

    prompt=f'Answer the following question {question}'

    answer=llm.invoke(prompt).content
    state['answer']=answer
    return state

In [28]:
graph=StateGraph(LLMState)

graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

workflow=graph.compile()

In [29]:
initial_state={'question':'Who is the creator of Python'}

final_state=workflow.invoke(initial_state)
print(final_state['answer'])

The creator of the Python programming language is **Guido van Rossum**.

He began developing Python in the late 1980s as a successor to the ABC programming language. The first public release of Python was in 1991. Guido van Rossum served as the project's principal author and maintainer for many years, often referred to as the "Benevolent Dictator For Life" (BDFL) of the Python community, until he stepped down from that role in 2018.
